# Imports

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Pinecone Upload

In [ ]:
# Load environment variables
load_dotenv()

# --- 1. CONFIGURATION ---
# Using gemini-embedding-001 (768 dimensions) as per official stable docs
INDEX_NAME = "youtube-agent-musical"
JSON_PATH = "../data/agent_db/godzilla_master_context.json"

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    google_api_key=os.getenv("GEMINI_API_KEY"),
    task_type="retrieval_document" # Optimized for storage/indexing
)

print(f"🚀 Initializing ingestion for index: {INDEX_NAME}")

try:
    # --- 2. DATA PREPARATION ---
    # Loading the refined master context (Demucs + Whisper logic)
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        agent_data = json.load(f)
    
    documents = []
    for segment in agent_data:
        text = segment.get("text", "").strip()
        
        # Only process segments with actual content
        if text:
            doc = Document(
                page_content=text,
                metadata={
                    "start": segment["start"],
                    "end": segment["end"],
                    "is_music_piece": segment["is_music_piece"]
                }
            )
            documents.append(doc)
            
    print(f"📄 Prepared {len(documents)} documents for vectorization.")

    # --- 3. PINECONE UPLOAD ---
    # Syncing vectors to the 768-dimension Gemini-compatible index
    vectorstore = PineconeVectorStore.from_documents(
        documents, 
        embedding=embeddings,
        index_name=INDEX_NAME,
        pinecone_api_key=os.getenv("PINECONE_API_KEY")
    )
    
    print("-" * 50)
    print("✅ SUCCESS: Ingestion complete with stable gemini-embedding-001!")
    print(f"Knowledge base is live in index: {INDEX_NAME}")
    print("-" * 50)

except Exception as e:
    print(f"❌ Critical Error during ingestion: {e}")

📄 Se han preparado 56 documentos para subir.
✅ Ingestion complete with stable gemini-embedding-001!


# Pinecone Search

In [ ]:
# Load environment variables
load_dotenv()

# --- 1. CONFIGURATION ---
# Target index: youtube-agent-musical (768 dimensions)
INDEX_NAME = "youtube-agent-musical"

# Using gemini-embedding-001 as the stable backbone for search
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001", 
    google_api_key=os.getenv("GEMINI_API_KEY"),
    task_type="retrieval_query" # Optimized for semantic search queries
)

# 2. CONNECT: Initialize the Vector Store connection
# We use the existing index filled during the ingestion phase
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME, 
    pinecone_api_key=os.getenv("PINECONE_API_KEY"),
    embedding=embeddings
)

# --- 3. THE AGENT TOOL ---
@tool
def search_video_knowledge(query: str, music_only: bool = False) -> str:
    """
    Searches the video transcript for relevant information to answer user questions.
    Set music_only=True if the user specifically asks about music, beats, or songs.
    """
    print(f"\n🔍 Agent is searching Pinecone for: '{query}' (Music Only: {music_only})")
    
    # Retrieve top 4 most relevant chunks
    search_kwargs = {"k": 4} 
    
    # Apply the Demucs-based music filter if requested
    if music_only:
        search_kwargs["filter"] = {"is_music_piece": True}
        
    # Execute semantic similarity search
    results = vectorstore.similarity_search(query, **search_kwargs)
    
    if not results:
        return "❌ I couldn't find any information about that in the video."
    
    # Format metadata and text for LLM consumption
    formatted_context = ""
    for i, doc in enumerate(results):
        start_time = doc.metadata.get("start", 0)
        end_time = doc.metadata.get("end", 0)
        is_music = doc.metadata.get("is_music_piece", False)
        
        tag = "🎵 [MUSIC PLAYING]" if is_music else "🗣️ [NARRATIVE]"
        formatted_context += f"--- Result {i+1} ---\n"
        formatted_context += f"Time

✅ Ingestion complete with stable gemini-embedding-001

Test 1: General Query

🔍 Agent is searching Pinecone for: 'What does Eminem do with a chainsaw?' (Music Only: False)
--- Result 1 ---
Time: 45.12s to 48.24s | 🗣️ [NARRATIVE]
Transcript: Slim Shady, where he's surrounded by Eminem doppelgangers.

--- Result 2 ---
Time: 110.08s to 114.36s | 🗣️ [NARRATIVE]
Transcript: Later, we find the rapper buttering meat with a chainsaw, and 10 years ago, Emmys to

--- Result 3 ---
Time: 110.08s to 114.36s | 🗣️ [NARRATIVE]
Transcript: Later, we find the rapper buttering meat with a chainsaw, and 10 years ago, Emmys to

--- Result 4 ---
Time: 45.12s to 48.24s | 🗣️ [NARRATIVE]
Transcript: Slim Shady, where he's surrounded by Eminem doppelgangers.


